In [ ]:
first = True

# Detecção de Gestos e Classificação em Libras

In [ ]:

if first:
    !pip install torch 
    !pip install torchvision 
    !pip install pandas
    !pip install av
    !pip install seaborn
    !pip install matplotlib
    !pip install beautifulsoup4
    !pip install opencv-python
    !pip install numpy
    !pip install yt-dlp
    !pip install mediapipe
    !pip install kagglehub
first = False

import torchvision.models as models
import torch.optim as optim
import kagglehub
import pandas as pd 
import seaborn as sns 
import numpy as np 
import matplotlib.pyplot as plt 
import torch 
from torch import nn
import cv2 
from PIL import Image
from bs4 import BeautifulSoup
import requests
import yt_dlp
from torchvision import models
import mediapipe as mp
import os 

## Download das Ferramentas
Temos que por se tratar de um algoritmo de aprendizado de máquina que vai ser usado em sistemas reais, temos que 
primeiramente, executar seu treinamento de forma que ele possa ser usado. Para isso, temos que o **V-Librasil** é um 
dataset público de libras que pode ser usado para criar sistemas capazes de entender a libras, e que pode ser usado para o treinamento 
de sistemas de aprendizado de máquina para essa linguagem

In [ ]:
INES_DICT = "https://dicionario.ines.gov.br/"
 
# ─── Dataset ──────────────────────────────────────────────────────────────────
print("Downloading vlibrasil dataset")
path = kagglehub.dataset_download("davimedio01/v-librasil")
print("Path to dataset files:", path)
 
videos_dir_path = path + "/videos UFPE (V-LIBRASIL)"
print("Path to videos directory:", videos_dir_path)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
 # ─── Carrega caminhos e labels ─────────────────────────────────────────────────
video_paths = []
label_list = []
 
class_names = sorted(os.listdir(videos_dir_path))
 
for i, class_name in enumerate(class_names):
    class_dir = os.path.join(videos_dir_path, class_name)
    if os.path.isdir(class_dir):
        for video_file in os.listdir(class_dir):
            if video_file.endswith(('.mp4', '.avi', '.mov')):
                video_paths.append(os.path.join(class_dir, video_file))
                label_list.append(i)
 
num_classes = len(class_names)
print(f"Found {len(video_paths)} videos, {num_classes} classes")
 
 
# ─── Função auxiliar: lê vídeo com PyAV (suporta interlaced) ──────────────────
def read_video_pyav(video_path, num_frames):
    """
    Lê frames de um vídeo usando PyAV.
    Suporta vídeos entrelaçados (interlaced) que o OpenCV não consegue ler.
    Retorna lista de arrays numpy (H, W, 3) em RGB.
    """
    frames = []
    try:
        container = av.open(video_path)
 
        for frame in container.decode(video=0):
            # to_ndarray com format="rgb24" faz o desentrelaçamento automaticamente
            frame_rgb = frame.to_ndarray(format="rgb24")
 
            frame_resized = cv2.resize(frame_rgb, (224, 224))
            frames.append(frame_resized)
 
            if len(frames) >= num_frames:
                break
 
        container.close()
 
    except Exception as e:
        print(f"Erro ao abrir {video_path}: {e}")
 
    # Padding: repete o último frame se o vídeo for mais curto que num_frames
    if len(frames) == 0:
        frames = [np.zeros((224, 224, 3), dtype=np.uint8)]
 
    while len(frames) < num_frames:
        frames.append(frames[-1])
 
    return frames[:num_frames]
 

Path to dataset files: /home/vmoura/.cache/kagglehub/datasets/davimedio01/v-librasil/versions/1
Path to videos directory: /home/vmoura/.cache/kagglehub/datasets/davimedio01/v-librasil/versions/1/videos UFPE (V-LIBRASIL)
Using device: cpu
Found 4086 videos, 3 classes


In [ ]:
# ─── VideoDataset ──────────────────────────────────────────────────────────────
class VideoDataset(torch.utils.data.Dataset):
    def __init__(self, video_paths, labels, num_frames, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.num_frames = num_frames
        self.transform = transform
 
    def __len__(self):
        return len(self.video_paths)
 
    def __getitem__(self, idx):
        frames = read_video_pyav(self.video_paths[idx], self.num_frames)
 
        video_tensor = torch.stack([
            torch.from_numpy(f).permute(2, 0, 1) for f in frames
        ])
        video_tensor = video_tensor.float() / 255.0
 
        return video_tensor, self.labels[idx]
 

In [ ]:
# ─── Modelo ────────────────────────────────────────────────────────────────────
class LibrasClassifier(nn.Module):
    def __init__(self, hidden_size, num_layers, num_classes):
        super(LibrasClassifier, self).__init__()
 
        self.feature_extractor = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.feature_extractor.fc = nn.Identity()
 
        self.lstm = nn.LSTM(512, hidden_size, num_layers, batch_first=True)
        self.classifier = nn.Linear(hidden_size, num_classes)
 
    def forward(self, x):
        batch_size, time_steps, C, H, W = x.size()
 
        x = x.view(batch_size * time_steps, C, H, W)
        features = self.feature_extractor(x)           # [Batch*Time, 512]
        features = features.view(batch_size, time_steps, -1)
 
        out, (hn, cn) = self.lstm(features)
        out = self.classifier(out[:, -1, :])
 
        return out

In [ ]:
# ─── Dataloader ────────────────────────────────────────────────────────────────
dataset = VideoDataset(video_paths, label_list, num_frames=30)
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,       # Leitura paralela de vídeos
    pin_memory=True      # Acelera transferência CPU → GPU
)
 

In [ ]:
# ─── Treinamento ───────────────────────────────────────────────────────────────
model = LibrasClassifier(
    hidden_size=256,
    num_layers=2,
    num_classes=num_classes
).to(device)
 
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
 
num_epochs = 10
 
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
 
    for videos, labels_batch in dataloader:
        videos = videos.to(device)
        labels_batch = labels_batch.to(device)
 
        optimizer.zero_grad()
 
        outputs = model(videos)
        loss = criterion(outputs, labels_batch)
 
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item()
 
    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}")
 

[swscaler @ 0x555c8e4fb800] Cannot convert interlaced to progressive frames or vice versa.
 (Invalid argument): fmt:yuv420p csp:bt709 prim:bt709 trc:bt709 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x555c8e4fb800] Cannot convert interlaced to progressive frames or vice versa.
 (Invalid argument): fmt:yuv420p csp:bt709 prim:bt709 trc:bt709 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x555c8e4fb800] Cannot convert interlaced to progressive frames or vice versa.
 (Invalid argument): fmt:yuv420p csp:bt709 prim:bt709 trc:bt709 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x555c8e4fb800] Cannot convert interlaced to progressive frames or vice versa.
 (Invalid argument): fmt:yuv420p csp:bt709 prim:bt709 trc:bt709 -> fmt:bgr24 csp:gbr prim:unknown trc:unknown
[swscaler @ 0x555c8e4fb800] Cannot convert interlaced to progressive frames or vice versa.
 (Invalid argument): fmt:yuv420p csp:bt709 prim:bt709 trc:bt709 -> fmt:bgr24 csp:gbr prim:unknown trc:unk